# 🔧 LLVM IR Generator & Validator
> Powered by Groq LLaMA · Validates via `llvm-as` + `opt --verify` · Interactive UI

**Workflow:** Enter a seed → Generate IR → Review output → Validate → See result

In [ ]:
#@title ⚙️ Cell 1 — Install Dependencies (run once)
%%capture
!apt-get update -qq
!apt-get install -y -qq clang llvm
!pip install -q groq ipywidgets

# Verify installs
import subprocess
tools = ["clang", "llvm-as", "opt", "lli"]
for t in tools:
    r = subprocess.run([t, "--version"], capture_output=True, text=True)
    ver = r.stdout.splitlines()[0] if r.stdout else r.stderr.splitlines()[0]
    print(f"✅ {t}: {ver}")

In [ ]:
#@title 🔑 Cell 2 — Set Groq API Key
import os
from getpass import getpass
from IPython.display import display, HTML

# Try to read from Colab Secrets first (best practice)
try:
    from google.colab import userdata
    api_key = userdata.get('GROQ_API_KEY')
    if api_key:
        os.environ["GROQ_API_KEY"] = api_key
        display(HTML('<p style="color:#2ecc71;font-weight:bold">✅ Loaded GROQ_API_KEY from Colab Secrets.</p>'))
    else:
        raise KeyError()
except Exception:
    os.environ["GROQ_API_KEY"] = getpass("🔑 Enter your Groq API Key: ")
    display(HTML('<p style="color:#2ecc71;font-weight:bold">✅ API key set.</p>'))

In [ ]:
#@title 🧠 Cell 3 — Core Generator & Validator Logic
import re
import os
import subprocess
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

# ── System Prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """\
You are a senior LLVM IR engineer. Your ONLY output is raw LLVM IR text.

Strict rules:
- Use LLVM 14 syntax exclusively.
- Always include `target datalayout` and `target triple`.
- Use typed pointers (no opaque pointers — avoid `ptr` keyword).
- All SSA registers must be defined before use.
- Every basic block must end with a terminator (ret, br, unreachable).
- Do NOT emit markdown, backticks, or any prose. Pure IR only.
"""

# ── Few-shot example ──────────────────────────────────────────────────────────
FEW_SHOT = r"""
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-f80:128-n8:16:32:64-S128"
target triple = "x86_64-pc-linux-gnu"

@fmt_int  = private unnamed_addr constant [4 x i8] c"%d\0A\00", align 1
@fmt_str  = private unnamed_addr constant [4 x i8] c"%s\0A\00", align 1

declare i32 @printf(i8* nocapture readonly, ...)

define i32 @add(i32 %a, i32 %b) {
entry:
  %result = add nsw i32 %a, %b
  ret i32 %result
}

define i32 @main() {
entry:
  %x = alloca i32, align 4
  %y = alloca i32, align 4
  store i32 15, i32* %x, align 4
  store i32 27, i32* %y, align 4
  %xval = load i32, i32* %x, align 4
  %yval = load i32, i32* %y, align 4
  %sum  = call i32 @add(i32 %xval, i32 %yval)
  %fmtptr = getelementptr inbounds [4 x i8], [4 x i8]* @fmt_int, i64 0, i64 0
  call i32 (i8*, ...) @printf(i8* %fmtptr, i32 %sum)
  ret i32 0
}
"""

# ── Generator ─────────────────────────────────────────────────────────────────
def generate_llvm_ir(seed: str, temperature: float = 0.4, max_tokens: int = 1500) -> str:
    """Call Groq and return clean LLVM IR text."""
    prompt = f"""\
Reference LLVM IR (LLVM 14, x86_64):

{FEW_SHOT}

Generate a NEW, complete, valid LLVM IR program with the theme: "{seed}"

Requirements:
1. Include target datalayout and target triple.
2. Use at least one function besides @main.
3. Demonstrate: arithmetic operations, a conditional branch (icmp + br), alloca/load/store, printf.
4. Return 0 from @main.
5. Output ONLY raw LLVM IR — no markdown, no backticks, no comments outside IR syntax.
"""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system",  "content": SYSTEM_PROMPT},
            {"role": "user",    "content": prompt},
        ],
    )
    raw = response.choices[0].message.content or ""
    # Strip any accidental markdown fences
    raw = re.sub(r"```[a-zA-Z]*", "", raw)
    raw = raw.replace("```", "")
    return raw.strip()


# ── Validator ─────────────────────────────────────────────────────────────────
def validate_ir(path: str) -> tuple[bool, str, str]:
    """
    Returns (is_valid, short_status, detailed_message).
    Runs llvm-as (syntax) then opt --verify (semantics).
    """
    # Step 1 — Syntax check via llvm-as
    r1 = subprocess.run(
        ["llvm-as", path, "-o", "/dev/null"],
        capture_output=True, text=True
    )
    if r1.returncode != 0:
        return False, "SYNTAX ERROR", r1.stderr.strip()

    # Step 2 — Semantic verification via opt
    r2 = subprocess.run(
        ["opt", "-verify", "-disable-output", path],
        capture_output=True, text=True
    )
    if r2.returncode != 0:
        return False, "SEMANTIC ERROR", r2.stderr.strip()

    return True, "VALID", "All checks passed (llvm-as + opt --verify)."


# ── Optional runner ───────────────────────────────────────────────────────────
def run_ir(path: str, timeout: int = 10) -> tuple[bool, str]:
    """Execute IR with lli and capture stdout/stderr."""
    r = subprocess.run(
        ["lli", path],
        capture_output=True, text=True, timeout=timeout
    )
    output = (r.stdout + r.stderr).strip()
    return r.returncode == 0, output

print("✅ Core logic loaded.")

In [ ]:
#@title 🖥️ Cell 4 — Interactive UI  ← Run this cell!
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── CSS ───────────────────────────────────────────────────────────────────────
display(HTML("""
<style>
  .llvm-card {
    border-radius: 10px;
    padding: 14px 18px;
    margin: 8px 0;
    font-family: monospace;
    font-size: 13px;
    line-height: 1.6;
    white-space: pre-wrap;
    word-break: break-word;
  }
  .llvm-ir-box   { background:#1e1e2e; color:#cdd6f4; border:1px solid #45475a; }
  .llvm-valid    { background:#1c3a2a; color:#a6e3a1; border:1px solid #40a02b; }
  .llvm-invalid  { background:#3a1c1c; color:#f38ba8; border:1px solid #d20f39; }
  .llvm-run-out  { background:#1e2a3a; color:#89dceb; border:1px solid #04a5e5; }
  .llvm-label    { font-family:sans-serif; font-weight:700; font-size:14px; margin-bottom:4px; }
  .llvm-badge-ok   { display:inline-block; background:#40a02b; color:#fff; padding:2px 10px; border-radius:12px; font-size:12px; font-family:sans-serif; }
  .llvm-badge-err  { display:inline-block; background:#d20f39; color:#fff; padding:2px 10px; border-radius:12px; font-size:12px; font-family:sans-serif; }
  .llvm-section  { margin-top: 16px; }
</style>
"""))

# ── Widgets ───────────────────────────────────────────────────────────────────
seed_input = widgets.Text(
    value="fibonacci sequence with loops",
    placeholder="e.g. bubble sort, factorial recursion, matrix multiply …",
    description="Seed:",
    layout=widgets.Layout(width="70%"),
    style={"description_width": "50px"},
)

temp_slider = widgets.FloatSlider(
    value=0.4, min=0.0, max=1.0, step=0.05,
    description="Temp:",
    readout_format=".2f",
    layout=widgets.Layout(width="40%"),
    style={"description_width": "50px"},
)

gen_btn      = widgets.Button(description="⚡ Generate IR",    button_style="primary",   layout=widgets.Layout(width="160px", height="38px"))
validate_btn = widgets.Button(description="🔍 Validate IR",    button_style="warning",   layout=widgets.Layout(width="160px", height="38px"))
run_btn      = widgets.Button(description="▶ Run with lli",   button_style="success",   layout=widgets.Layout(width="160px", height="38px"))
clear_btn    = widgets.Button(description="🗑 Clear",           button_style="danger",    layout=widgets.Layout(width="100px", height="38px"))

# Disable validate/run until IR is generated
validate_btn.disabled = True
run_btn.disabled      = True

status_bar = widgets.HTML(value="<i style='color:#888'>Ready. Enter a seed and click Generate.</i>")
out        = widgets.Output()

# State shared between callbacks
_state = {"ir_path": None, "ir_text": None}

# ── Helpers ───────────────────────────────────────────────────────────────────
def _html_ir(ir_text):
    safe = ir_text.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
    return f"""<div class='llvm-section'>
  <div class='llvm-label'>📄 Generated LLVM IR</div>
  <div class='llvm-card llvm-ir-box'>{safe}</div>
</div>"""

def _html_validation(is_valid, status, detail):
    badge  = f"<span class='llvm-badge-ok'>✔ {status}</span>" if is_valid else f"<span class='llvm-badge-err'>✘ {status}</span>"
    cls    = "llvm-valid" if is_valid else "llvm-invalid"
    safe_d = detail.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
    return f"""<div class='llvm-section'>
  <div class='llvm-label'>🔍 Validation Result &nbsp; {badge}</div>
  <div class='llvm-card {cls}'>{safe_d}</div>
</div>"""

def _html_run(success, output):
    safe = output.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;") or "<no output>"
    badge = "<span class='llvm-badge-ok'>exit 0</span>" if success else "<span class='llvm-badge-err'>non-zero exit</span>"
    return f"""<div class='llvm-section'>
  <div class='llvm-label'>▶ Execution Output &nbsp; {badge}</div>
  <div class='llvm-card llvm-run-out'>{safe}</div>
</div>"""

# ── Callbacks ─────────────────────────────────────────────────────────────────
def on_generate(_):
    seed = seed_input.value.strip()
    if not seed:
        status_bar.value = "<span style='color:#f38ba8'>⚠ Please enter a seed description.</span>"
        return

    gen_btn.disabled      = True
    validate_btn.disabled = True
    run_btn.disabled      = True
    status_bar.value      = "<span style='color:#89b4fa'>⏳ Generating IR from Groq …</span>"

    with out:
        clear_output(wait=True)
        try:
            ir = generate_llvm_ir(seed, temperature=temp_slider.value)
            path = "/tmp/generated.ll"
            with open(path, "w") as f:
                f.write(ir)
            _state["ir_path"] = path
            _state["ir_text"] = ir
            display(HTML(_html_ir(ir)))
            status_bar.value      = "<span style='color:#a6e3a1'>✅ IR generated. Review above, then click <b>Validate IR</b>.</span>"
            validate_btn.disabled = False
            run_btn.disabled      = False
        except Exception as e:
            display(HTML(f"<div class='llvm-card llvm-invalid'>❌ Generation failed:\n{e}</div>"))
            status_bar.value = "<span style='color:#f38ba8'>❌ Generation error.</span>"
        finally:
            gen_btn.disabled = False


def on_validate(_):
    path = _state.get("ir_path")
    if not path:
        status_bar.value = "<span style='color:#f38ba8'>⚠ Generate IR first.</span>"
        return

    validate_btn.disabled = True
    status_bar.value      = "<span style='color:#89b4fa'>⏳ Validating …</span>"

    with out:
        # Keep the IR display, append validation below it
        display(HTML(_html_ir(_state["ir_text"])))
        try:
            is_valid, status, detail = validate_ir(path)
            display(HTML(_html_validation(is_valid, status, detail)))
            if is_valid:
                status_bar.value = "<span style='color:#a6e3a1'>✅ IR is VALID. You can run it with ▶ Run with lli.</span>"
            else:
                status_bar.value = "<span style='color:#f38ba8'>❌ IR is INVALID. See details above. Try regenerating.</span>"
        except Exception as e:
            display(HTML(f"<div class='llvm-card llvm-invalid'>❌ Validation error:\n{e}</div>"))
            status_bar.value = "<span style='color:#f38ba8'>❌ Validation error.</span>"
        finally:
            validate_btn.disabled = False


def on_run(_):
    path = _state.get("ir_path")
    if not path:
        return

    run_btn.disabled = True
    status_bar.value = "<span style='color:#89b4fa'>⏳ Running with lli …</span>"

    with out:
        display(HTML(_html_ir(_state["ir_text"])))
        try:
            success, output = run_ir(path)
            display(HTML(_html_run(success, output)))
            status_bar.value = "<span style='color:#a6e3a1'>✅ Execution complete.</span>" if success else \
                               "<span style='color:#fab387'>⚠ Program exited with non-zero code.</span>"
        except subprocess.TimeoutExpired:
            display(HTML("<div class='llvm-card llvm-invalid'>⏱ Execution timed out (10 s).</div>"))
            status_bar.value = "<span style='color:#f38ba8'>⏱ Timed out.</span>"
        except Exception as e:
            display(HTML(f"<div class='llvm-card llvm-invalid'>❌ Run error:\n{e}</div>"))
        finally:
            run_btn.disabled = False


def on_clear(_):
    _state["ir_path"] = None
    _state["ir_text"] = None
    validate_btn.disabled = True
    run_btn.disabled      = True
    status_bar.value      = "<i style='color:#888'>Cleared. Enter a new seed.</i>"
    with out:
        clear_output()


gen_btn.on_click(on_generate)
validate_btn.on_click(on_validate)
run_btn.on_click(on_run)
clear_btn.on_click(on_clear)

# ── Layout ────────────────────────────────────────────────────────────────────
display(HTML("<h3 style='font-family:sans-serif;margin-bottom:4px'>🔧 LLVM IR Generator & Validator</h3>"))
display(widgets.HBox([seed_input]))
display(widgets.HBox([temp_slider]))
display(widgets.HBox([gen_btn, validate_btn, run_btn, clear_btn], layout=widgets.Layout(gap="8px", margin="8px 0")))
display(status_bar)
display(out)

In [ ]:
#@title 📊 Cell 5 — Batch Test (optional)
#@markdown Run multiple seeds and get a pass/fail summary table.

SEEDS = [
    "fibonacci sequence with a loop",
    "bubble sort on a fixed array",
    "factorial using recursion",
    "counting even numbers 1–20",
    "GCD using Euclidean algorithm",
]

from IPython.display import display, HTML

rows = []
stats = {"total": 0, "valid": 0, "run_ok": 0}

for i, seed in enumerate(SEEDS):
    stats["total"] += 1
    path = f"/tmp/batch_{i}.ll"

    # Generate
    try:
        ir = generate_llvm_ir(seed, temperature=0.3)
        with open(path, "w") as f:
            f.write(ir)
        gen_ok = True
    except Exception as e:
        rows.append((seed, "❌ gen failed", "—", str(e)[:60]))
        continue

    # Validate
    is_valid, status, detail = validate_ir(path)
    if is_valid:
        stats["valid"] += 1
        val_cell = "<span style='color:#a6e3a1'>✔ VALID</span>"
    else:
        val_cell = f"<span style='color:#f38ba8'>✘ {status}</span>"

    # Run
    if is_valid:
        try:
            ok, output = run_ir(path)
            run_cell = f"<span style='color:#89dceb'>exit {0 if ok else 1}: {output[:40]}</span>"
            if ok: stats["run_ok"] += 1
        except Exception as e:
            run_cell = f"<span style='color:#fab387'>⏱ {str(e)[:40]}</span>"
    else:
        run_cell = "—"

    rows.append((seed, val_cell, run_cell, detail[:60] if not is_valid else "—"))

# Build HTML table
table_rows = "".join(
    f"<tr><td style='padding:6px 12px'>{r[0]}</td>"
    f"<td style='padding:6px 12px;text-align:center'>{r[1]}</td>"
    f"<td style='padding:6px 12px'>{r[2]}</td>"
    f"<td style='padding:6px 8px;font-size:11px;color:#888'>{r[3]}</td></tr>"
    for r in rows
)
html = f"""
<h3 style='font-family:sans-serif'>📊 Batch Results — {stats['valid']}/{stats['total']} valid, {stats['run_ok']} ran OK</h3>
<table style='border-collapse:collapse;font-family:monospace;font-size:13px;width:100%'>
  <thead>
    <tr style='background:#313244;color:#cdd6f4'>
      <th style='padding:8px 12px;text-align:left'>Seed</th>
      <th style='padding:8px 12px'>Validation</th>
      <th style='padding:8px 12px;text-align:left'>Run output</th>
      <th style='padding:8px 8px;text-align:left'>Note</th>
    </tr>
  </thead>
  <tbody style='background:#1e1e2e;color:#cdd6f4'>
    {table_rows}
  </tbody>
</table>
"""
display(HTML(html))